
# Exp.02B — PGR²M τ Sensitivity: paper-style HumanML3D test evaluation

This notebook evaluates the four **best-validation-FID** checkpoints from Exp.02 (`τ = 0.1, 0.3, 0.7, 0.9`) using the repository's own `eval_pg_tokenizer.py`.

- 20 repeated HumanML3D test evaluations per checkpoint
- 95% CI exactly as the public script computes it: `1.96 * np.std(x) / sqrt(20)`
- Primary reviewer-facing metrics: **Top-1, FID, MM-Dist, Orthogonality**
- Supplemental metrics: Diversity, Top-2/3, MPJPE, PA-MPJPE, ACCEL
- No retraining
- Safe to re-run: already completed τ logs are skipped


**Fix in B3:** the pinned OpenAI CLIP checkout is kept, and the exact missing runtime dependency `ftfy==6.1.1` is installed into the Python 3 environment only if it is absent. No training/checkpoint files are modified.


In [ ]:

from pathlib import Path
import os, sys, json, re, time, hashlib, subprocess, shlex
import pandas as pd
import numpy as np

PGR = Path(".")
PY = Path("venv/bin/python")

TRAIN_ROOT = PGR / "exp_result_revision_tau_sensitivity"
RESULT_ROOT = PGR / "revision_results" / "exp02_tau"
VAL_AUDIT = RESULT_ROOT / "exp02_validation_audit.csv"

PRE = PGR / "revision_preflight"
LOGDIR = PGR / "revision_logs" / "exp02b_tau"
EVAL_OUT = PGR / "eval_output_revision_tau_sensitivity"

EVAL_SRC = PGR / "eval_pg_tokenizer.py"
COMPAT = PRE / "eval_pg_tokenizer_exp02b_compat.py"
SMOKE_SRC = PRE / "eval_pg_tokenizer_exp02b_smoke_1repeat.py"
SMOKE_COMPAT = PRE / "eval_pg_tokenizer_exp02b_smoke_compat.py"

FREEZE = RESULT_ROOT / "exp02b_test_eval_freeze.json"
RUNS_CSV = RESULT_ROOT / "exp02b_test_runs.csv"
SUMMARY_CSV = RESULT_ROOT / "exp02b_test_summary.csv"
PAPER_CSV = RESULT_ROOT / "exp02b_reviewer_table.csv"

for d in [RESULT_ROOT, PRE, LOGDIR, EVAL_OUT]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

print("=== EXP.02B ENVIRONMENT ===")
print("Python :", sys.executable)
print("Expected:", PY)
if Path(sys.executable).resolve() != PY.resolve():
    print("[WARN] Notebook kernel interpreter differs from Python 3 venv.")
    print("       Select kernel `Python 3` and restart the notebook before continuing.")
print("PGR    :", PGR)
print("Eval   :", EVAL_SRC)

assert PY.exists()
assert EVAL_SRC.exists()
assert VAL_AUDIT.exists()

val_audit = pd.read_csv(VAL_AUDIT)
display(val_audit)


In [ ]:

# 2. Discover the exact best-FID checkpoint folders and freeze provenance.

TAUS = [0.1, 0.3, 0.7, 0.9]

def tag(tau):
    return str(tau).replace(".", "p")

def discover_run_folder(tau):
    base = TRAIN_ROOT / f"exp_pg_tokenizer_tau_{tag(tau)}_seed123"
    candidates = []
    for ckpt in base.rglob("net_best_fid.pth"):
        if (ckpt.parent / "arguments.yaml").exists():
            candidates.append(ckpt.parent)

    if not candidates:
        raise RuntimeError(f"No best-FID checkpoint found for tau={tau}: {base}")

    candidates = sorted(
        candidates,
        key=lambda p: (p / "net_best_fid.pth").stat().st_mtime
    )

    if len(candidates) > 1:
        print(f"[WARN] tau={tau}: multiple run directories found")
        for x in candidates:
            print("   ", x)
        print("   -> selecting newest:", candidates[-1])

    return candidates[-1]

selected = {}
freeze_rows = []

for tau in TAUS:
    rd = discover_run_folder(tau)
    selected[tau] = rd

    vr = val_audit[np.isclose(val_audit["tau"].astype(float), tau)].iloc[0]
    ckpt = rd / "net_best_fid.pth"
    args = rd / "arguments.yaml"

    freeze_rows.append({
        "tau": tau,
        "run_dir": str(rd),
        "checkpoint": str(ckpt),
        "checkpoint_sha256": sha256(ckpt),
        "arguments_sha256": sha256(args),
        "best_val_fid_iter": int(vr["best_fid_improvement_iter"]),
        "best_val_fid_logged": float(vr["best_fid_logged_5dp"]),
        "orthogonality": float(vr["orthogonality"]),
    })

freeze_df = pd.DataFrame(freeze_rows)
display(freeze_df)

FREEZE.write_text(json.dumps({
    "protocol": "PGR2M public eval_pg_tokenizer.py; best-validation-FID checkpoint; 20 repeats",
    "ci_formula": "1.96 * np.std(x, ddof=0) / sqrt(20)",
    "eval_source_sha256": sha256(EVAL_SRC),
    "conditions": freeze_rows,
}, indent=2))

print("[OK] Freeze:", FREEZE)


In [ ]:

# 3. Create compatibility wrappers without modifying the public source.

compat_text = '''import os, sys, runpy
from pathlib import Path
import numpy as np

PGR = Path(".")
CLIP_ROOT = PGR / "revision_preflight" / "third_party" / "openai_clip_dcba3cb2e282"

os.chdir(PGR)
sys.path.insert(0, str(PGR))

# Exp.00S12 recovered and pinned the exact OpenAI CLIP checkout.
# utils/eval_trans.py imports `clip` transitively, so expose it here.
if not CLIP_ROOT.exists():
    raise RuntimeError(
        f"Recovered OpenAI CLIP checkout is missing: {CLIP_ROOT}. "
        "Do not install an unrelated PyPI package named `clip`."
    )
sys.path.insert(0, str(CLIP_ROOT))

try:
    import ftfy  # required by the pinned OpenAI CLIP tokenizer
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Missing ftfy in evaluator subprocess. Re-run notebook cell 3; "
        "it installs the pinned ftfy==6.1.1 dependency."
    ) from e

# Public utils/eval_trans.py imports two helper symbols that were not released
# with the public repository. They are unrelated to tokenizer evaluation_dec(),
# which is the only path used by eval_pg_tokenizer.py here. Add import-only
# placeholders so the public module can load. If another code path ever tries
# to call them, fail loudly instead of silently substituting behavior.
def _unreleased_subsamples_loader(*args, **kwargs):
    raise NotImplementedError(
        "get_subsamples_loader was not released in the public PGR2M dataset loaders "
        "and is not required by Exp.02B tokenizer evaluation_dec."
    )

import dataset.dataset_TM_eval as _tm_eval
if not hasattr(_tm_eval, "get_subsamples_loader"):
    _tm_eval.get_subsamples_loader = _unreleased_subsamples_loader

# The released repository may omit dataset.dataset_RTM_eval entirely, while
# utils/eval_trans.py imports it at module import time. Create an in-memory,
# import-only placeholder module. This does NOT change repository files.
import types as _types
import dataset as _dataset_pkg

try:
    import dataset.dataset_RTM_eval as _rtm_eval
    if not hasattr(_rtm_eval, "get_subsamples_loader"):
        _rtm_eval.get_subsamples_loader = _unreleased_subsamples_loader
except ModuleNotFoundError:
    _rtm_eval = _types.ModuleType("dataset.dataset_RTM_eval")
    _rtm_eval.get_subsamples_loader = _unreleased_subsamples_loader
    sys.modules["dataset.dataset_RTM_eval"] = _rtm_eval
    setattr(_dataset_pkg, "dataset_RTM_eval", _rtm_eval)

if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool

runpy.run_path(str(PGR / "eval_pg_tokenizer.py"), run_name="__main__")
'''
COMPAT.write_text(compat_text)
compile(COMPAT.read_text(), str(COMPAT), "exec")

# Verify/install the dependency using the SAME interpreter used for evaluation.
# This avoids importing ftfy into a stale notebook kernel by mistake.
dep_check = r"""
import importlib.util, subprocess, sys
from pathlib import Path

clip_root = Path("revision_preflight/third_party/openai_clip_dcba3cb2e282")
assert clip_root.exists(), f"Missing recovered CLIP checkout: {clip_root}"

if importlib.util.find_spec("ftfy") is None:
    print("[RUN] Installing pinned evaluator dependency: ftfy==6.1.1")
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", "ftfy==6.1.1", "--no-deps"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(p.stdout)
    if p.returncode != 0:
        raise SystemExit(p.returncode)

import ftfy
print("[OK] ftfy:", getattr(ftfy, "__version__", "unknown"), ftfy.__file__)

sys.path.insert(0, str(clip_root))
import clip
print("[OK] CLIP import:", clip.__file__)
"""

p = subprocess.run(
    [str(PY), "-c", dep_check],
    cwd=PGR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=1800,
)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError(
        "Pinned evaluator dependency preflight failed under Python 3 interpreter. "
        "See output above."
    )

src = EVAL_SRC.read_text()
needle = "repeat_time = 20"
if src.count(needle) != 1:
    raise RuntimeError(
        f"Expected exactly one `{needle}` in eval_pg_tokenizer.py, "
        f"found {src.count(needle)}"
    )

SMOKE_SRC.write_text(src.replace(needle, "repeat_time = 1", 1))

smoke_compat_text = compat_text.replace(
    'runpy.run_path(str(PGR / "eval_pg_tokenizer.py"), run_name="__main__")',
    f'runpy.run_path({str(SMOKE_SRC)!r}, run_name="__main__")'
)
SMOKE_COMPAT.write_text(smoke_compat_text)
compile(SMOKE_COMPAT.read_text(), str(SMOKE_COMPAT), "exec")

print("[OK] Import shim : TM/RTM unreleased helper imports guarded")
print("[OK] Final compat :", COMPAT)
print("[OK] Smoke compat :", SMOKE_COMPAT)


In [ ]:

# 4. One-repeat smoke gate on tau=0.1.
#    This smoke number is NOT a paper result.

def eval_cmd(script, tau, out_dir, exp_name):
    return [
        str(PY), "-u", str(script),
        "--batch-size", "256",
        "--lr", "2e-4",
        "--total-iter", "300000",
        "--lr-scheduler", "200000",
        "--nb-code", "392",
        "--down-t", "2",
        "--depth", "3",
        "--dilation-growth-rate", "3",
        "--out-dir", str(out_dir),
        "--dataname", "t2m",
        "--vq-act", "relu",
        "--exp-name", exp_name,
        "--use-keywords",
        "--dec-checkpoint-folder", str(selected[tau]),
        "--output-emb-width", "392",
        "--num_workers", "4",
        "--seed", "123",
    ]

smoke_log = LOGDIR / "exp02b_smoke_tau_0p1.log"
cmd = eval_cmd(SMOKE_COMPAT, 0.1, EVAL_OUT / "_smoke", "tau_0p1")

print("=== EXP.02B ONE-REPEAT SMOKE ===")
print(" ".join(shlex.quote(x) for x in cmd))

with open(smoke_log, "w") as f:
    p = subprocess.run(
        cmd,
        cwd=PGR,
        stdout=f,
        stderr=subprocess.STDOUT,
        timeout=7200
    )

print("returncode:", p.returncode)
print("\n".join(smoke_log.read_text(errors="replace").splitlines()[-80:]))

smoke_text = smoke_log.read_text(errors="replace")
if p.returncode != 0 or "final result:" not in smoke_text.lower():
    raise RuntimeError(f"Smoke failed. See {smoke_log}")

print("[OK] One-repeat test evaluation completed.")


In [ ]:

# 5. Run the four official 20-repeat test evaluations sequentially.
#    Safe to re-run: a tau log containing `final result:` is skipped.

jobs = []
for tau in TAUS:
    jobs.append({
        "tau": tau,
        "log": LOGDIR / f"exp02b_tau_{tag(tau)}.log",
        "cmd": eval_cmd(
            COMPAT,
            tau,
            EVAL_OUT,
            f"tau_{tag(tau)}"
        )
    })

print("=== EXP.02B FULL TEST EVALUATION ===")

for j in jobs:
    tau = j["tau"]
    log_path = j["log"]

    if log_path.exists():
        old_text = log_path.read_text(errors="replace")
        if "final result:" in old_text.lower():
            print(f"[SKIP] tau={tau}: completed log already exists")
            continue

    print(f"[RUN] tau={tau}")
    print("      log:", log_path)

    t0 = time.time()
    with open(log_path, "w") as f:
        p = subprocess.run(
            j["cmd"],
            cwd=PGR,
            stdout=f,
            stderr=subprocess.STDOUT
        )

    elapsed = (time.time() - t0) / 60
    txt = log_path.read_text(errors="replace")

    print(
        f"[END] tau={tau} rc={p.returncode} "
        f"final={'final result:' in txt.lower()} "
        f"elapsed={elapsed:.1f} min"
    )

    if p.returncode != 0 or "final result:" not in txt.lower():
        print("\n".join(txt.splitlines()[-100:]))
        raise RuntimeError(f"tau={tau} evaluation failed: {log_path}")

print("[OK] All four 20-repeat evaluations are complete.")


In [ ]:

# 6. Parse the 20 repeated test evaluations and build paper/reviewer tables.

# The logger emits a timestamped line plus an echo line.
# Restrict to timestamp-prefixed `Eva. Iter 0` lines to avoid duplicates.

pat = re.compile(
    r"^\d{4}-\d{2}-\d{2} .*?Eva\. Iter 0 :, "
    r"FID\. (?P<fid>[-+0-9.eE]+), "
    r"Diversity Real\. (?P<div_real>[-+0-9.eE]+), "
    r"Diversity\. (?P<div>[-+0-9.eE]+), "
    r"R_precision_real\. \[[^\]]+\], "
    r"R_precision\. \[(?P<rp>[^\]]+)\], "
    r"matching_score_real\. (?P<match_real>[-+0-9.eE]+), "
    r"matching_score_pred\. (?P<matching>[-+0-9.eE]+), "
    r"MPJPE:(?P<mpjpe>[-+0-9.eE]+), "
    r"PA-MPJPE:(?P<pampjpe>[-+0-9.eE]+), "
    r"ACCEL:(?P<accel>[-+0-9.eE]+)"
)

run_rows = []

for j in jobs:
    tau = j["tau"]
    log_path = j["log"]

    if not log_path.exists():
        raise RuntimeError(f"Missing log: {log_path}")

    matched = []

    for line in log_path.read_text(errors="replace").splitlines():
        m = pat.search(line)
        if not m:
            continue

        g = m.groupdict()
        rp = [float(x) for x in g["rp"].split()]

        if len(rp) != 3:
            raise RuntimeError(
                f"Unexpected R_precision at tau={tau}: {g['rp']}"
            )

        matched.append({
            "fid": float(g["fid"]),
            "diversity": float(g["div"]),
            "top1": rp[0],
            "top2": rp[1],
            "top3": rp[2],
            "matching": float(g["matching"]),
            "mpjpe": float(g["mpjpe"]),
            "pampjpe": float(g["pampjpe"]),
            "accel": float(g["accel"]),
        })

    if len(matched) != 20:
        raise RuntimeError(
            f"Expected exactly 20 timestamped test evaluations "
            f"for tau={tau}, got {len(matched)}. Inspect {log_path}"
        )

    for i, row in enumerate(matched, 1):
        run_rows.append({"tau": tau, "repeat": i, **row})

runs = pd.DataFrame(run_rows)
runs.to_csv(RUNS_CSV, index=False)

def ci95_public(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * np.std(x, ddof=0) / np.sqrt(len(x))

metrics = [
    "fid", "diversity",
    "top1", "top2", "top3",
    "matching",
    "mpjpe", "pampjpe", "accel"
]

summary_rows = []

for tau in TAUS:
    sub = runs[np.isclose(runs["tau"], tau)]
    vr = val_audit[np.isclose(
        val_audit["tau"].astype(float), tau
    )].iloc[0]

    row = {
        "tau": tau,
        "n": len(sub),
        "orthogonality": float(vr["orthogonality"]),
        "best_val_fid_iter": int(vr["best_fid_improvement_iter"]),
        "best_val_fid_logged": float(vr["best_fid_logged_5dp"]),
    }

    for metric in metrics:
        row[f"{metric}_mean"] = float(sub[metric].mean())
        row[f"{metric}_ci95"] = float(ci95_public(sub[metric]))

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_CSV, index=False)

def pm(mean, ci, digits=3):
    return f"{mean:.{digits}f} ± {ci:.{digits}f}"

paper = pd.DataFrame({
    "tau": summary["tau"],
    "Orthogonality ↓": summary["orthogonality"].map(
        lambda x: f"{x:.5f}"
    ),
    "Top-1 ↑": [
        pm(m, c, 3)
        for m, c in zip(summary["top1_mean"], summary["top1_ci95"])
    ],
    "FID ↓": [
        pm(m, c, 3)
        for m, c in zip(summary["fid_mean"], summary["fid_ci95"])
    ],
    "MM-Dist ↓": [
        pm(m, c, 3)
        for m, c in zip(summary["matching_mean"], summary["matching_ci95"])
    ],
    "MPJPE ↓": [
        pm(m, c, 4)
        for m, c in zip(summary["mpjpe_mean"], summary["mpjpe_ci95"])
    ],
    "PA-MPJPE ↓": [
        pm(m, c, 4)
        for m, c in zip(summary["pampjpe_mean"], summary["pampjpe_ci95"])
    ],
    "ACCEL ↓": [
        pm(m, c, 5)
        for m, c in zip(summary["accel_mean"], summary["accel_ci95"])
    ],
})

paper.to_csv(PAPER_CSV, index=False)

print("=== EXP.02B TEST SUMMARY ===")
display(summary)

print("\n=== REVIEWER-FACING TABLE ===")
display(paper)

print("\nSaved:")
print(" ", RUNS_CSV)
print(" ", SUMMARY_CSV)
print(" ", PAPER_CSV)


In [ ]:

# 7. Final decision.

for p in [RUNS_CSV, SUMMARY_CSV, PAPER_CSV]:
    assert p.exists(), p

runs = pd.read_csv(RUNS_CSV)
summary = pd.read_csv(SUMMARY_CSV)

ok_n = all(
    len(runs[np.isclose(runs["tau"], tau)]) == 20
    for tau in TAUS
)
ok_finite = np.isfinite(
    summary.select_dtypes(include=[np.number]).to_numpy()
).all()
ok_tau = np.allclose(
    sorted(summary["tau"].tolist()),
    TAUS
)

print("=== EXP.02B FINAL STATUS ===")
print("20 repeats / condition :", ok_n)
print("All numeric finite     :", ok_finite)
print("All tau present        :", ok_tau)
print("Summary                :", SUMMARY_CSV)
print("Reviewer table         :", PAPER_CSV)

FINALIZED = bool(ok_n and ok_finite and ok_tau)
print("FINALIZED              :", FINALIZED)

if not FINALIZED:
    raise RuntimeError("Exp.02B is not finalized.")

print("\nDECISION: EXP.02B TEST EVALUATION IS COMPLETE.")
print("Next: interpret the tau trade-off and insert the reviewer-facing table/response text.")


In [ ]:
import pandas as pd

summary = pd.read_csv(
    "revision_results/exp02_tau/exp02b_test_summary.csv"
)
reviewer = pd.read_csv(
    "revision_results/exp02_tau/exp02b_reviewer_table.csv"
)

print("=== EXP.02B TEST SUMMARY ===")
display(summary)

print("\n=== REVIEWER-FACING TABLE ===")
display(reviewer)